# 03 - Checking Data Consistency

The purpose of this document is to ensure there is **no Noisy Data** in the interim dataset as the output of the *ETL.py* script.

## Setting Up Project Directory

In [49]:
from jupyter_init import setup

setup()

from notebooks.logging_config import MyLogger
from src_code.config import *
from notebooks.constants import LOG_DIR as NTB_LOG_DIR

logger = MyLogger(label="Data Cleaning", section_name="data cleaning", file_log_path=NTB_LOG_DIR / "data_cleanness.log")

## Loading Dataset

In [50]:
import pandas as pd
import numpy as np

from src_code.versioning import VersionedFileManager

# DF_PATH = INTERIM_DATA_DIR / 'train_labeled_features_partial_v1.feather'
df_versioner = VersionedFileManager(file_path=INTERIM_DATA_DIR / "train_labeled_features_partial.feather", logger=logger)

# ---- LOAD ----
df = pd.read_feather(df_versioner.current_newest)
print(f"Loaded dataframe with {len(df)} rows and {len(df.columns)} columns\n")

df.dtypes

# type(df['filepath'])

[Data Cleaning RESULT] Current newest version: C:\Users\fmojt\Code\DPThesis\DP_Thesis\data\interim\train_labeled_features_partial_v26.feather
Loaded dataframe with 58294 rows and 30 columns



repo                                          str
commit                                        str
lines                                      object
content                                       str
filepath                                   object
author_email                                  str
canonical_datetime            datetime64[us, UTC]
author_exp_pre                              int64
author_recent_activity_pre                  int64
has_bug                                      bool
label                                       int64
loc_added                                   int64
loc_deleted                                 int64
files_changed                               int64
hunks_count                                 int64
msg_len                                     int64
has_fix_kw                                  int64
has_bug_kw                                  int64
ast_delta                                   int64
complexity_delta                            int64


## Converting NumpyArray -> List



In [51]:
#Convert the NumPy arrays back to Python lists
for col in ['code_embed', 'msg_embed', 'lines']:
    # Use .apply(list) or .apply(lambda x: x.tolist()) for robustness
    df[col] = df[col].apply(list)


## Missing Value Audit

In [52]:
print("## 1. Missing Values per Column")
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls.to_markdown())

## 1. Missing Values per Column
|                            |   0 |
|:---------------------------|----:|
| repo                       |   0 |
| commit                     |   0 |
| lines                      |   0 |
| content                    |   0 |
| filepath                   |   0 |
| author_email               |   0 |
| canonical_datetime         |   0 |
| author_exp_pre             |   0 |
| author_recent_activity_pre |   0 |
| has_bug                    |   0 |
| label                      |   0 |
| loc_added                  |   0 |
| loc_deleted                |   0 |
| files_changed              |   0 |
| hunks_count                |   0 |
| msg_len                    |   0 |
| has_fix_kw                 |   0 |
| has_bug_kw                 |   0 |
| ast_delta                  |   0 |
| complexity_delta           |   0 |
| max_func_change            |   0 |
| time_since_last_change     |   0 |
| todo                       |   0 |
| fixme                      |   0 |
| try 

## Primary Key Integrity

In [53]:
print("## 2. Primary Key Uniqueness Check")
key_cols = ["repo", "commit"]

dupes = df.duplicated(subset=key_cols).sum()
print(f"Duplicate key rows: {dupes}")

## 2. Primary Key Uniqueness Check
Duplicate key rows: 0


## Label Distribution

In [54]:
print("## 3. Label Distribution")
print(df['label'].value_counts(normalize=True).to_markdown())

## 3. Label Distribution
|   label |   proportion |
|--------:|-------------:|
|       0 |     0.658764 |
|       1 |     0.341236 |


## Repository Distribution (Imbalance Check)

In [55]:
print("## 4. Repository Distribution")
repo_dist = df['repo'].value_counts(normalize=True)
print(repo_dist.to_markdown())

## 4. Repository Distribution
| repo    |   proportion |
|:--------|-------------:|
| ansible |   0.386815   |
| sentry  |   0.235547   |
| core    |   0.164339   |
| pandas  |   0.100714   |
| ray     |   0.0595945  |
| airflow |   0.045099   |
| numpy   |   0.00789104 |


## Value Range Scan for Numeric Columns

Automatically detects:

- negatives where not allowed
- max values
- suspicious spikes

In [56]:
print("## 5. Numeric Column Range Scan")
num_cols = df.select_dtypes(include=[np.number]).columns

ranges = pd.DataFrame({
    "min": df[num_cols].min(),
    "median": df[num_cols].median(),
    "mean": df[num_cols].mean(),
    "max": df[num_cols].max()
})

print(ranges.to_markdown())

## 5. Numeric Column Range Scan
|                            |   min |   median |          mean |              max |
|:---------------------------|------:|---------:|--------------:|-----------------:|
| author_exp_pre             |     0 |       67 | 442.167       |   9425           |
| author_recent_activity_pre |     0 |        7 |  28.5873      |    667           |
| label                      |     0 |        0 |   0.341236    |      1           |
| loc_added                  |     0 |        4 |  65.836       | 811654           |
| loc_deleted                |     0 |       16 |  80.5227      |  29583           |
| files_changed              |     1 |        2 |   4.03767     |   4183           |
| hunks_count                |     0 |        8 |  17.5805      |   8315           |
| msg_len                    |     1 |       73 | 154.776       |  13182           |
| has_fix_kw                 |     0 |        0 |   0.335952    |      1           |
| has_bug_kw                 |   

## Check Columns Expected to Be Non-Negative

In [57]:
non_negative_cols = [
    "loc_added", "loc_deleted",
    "files_changed", "hunks_count",
    "msg_len", "ast_delta",
    "complexity_delta", "max_func_change",
    "author_exp_pre", "author_recent_activity_pre",
    "todo", "fixme", "try", "except", "raise",
    "recent_churn"
]

print("## 6. Negative Value Check")
for col in non_negative_cols:
    bad = (df[col] < 0).sum()
    print(f"{col}: {bad} negative values")

## 6. Negative Value Check
loc_added: 0 negative values
loc_deleted: 0 negative values
files_changed: 0 negative values
hunks_count: 0 negative values
msg_len: 0 negative values
ast_delta: 0 negative values
complexity_delta: 0 negative values
max_func_change: 0 negative values
author_exp_pre: 0 negative values
author_recent_activity_pre: 0 negative values
todo: 0 negative values
fixme: 0 negative values
try: 0 negative values
except: 0 negative values
raise: 0 negative values
recent_churn: 0 negative values


## Suspicious Feature Check: time_since_last_change

In [58]:
print("## 7. time_since_last_change Outliers")
tslc = df["time_since_last_change"]

print(f"Negative values: {(tslc < 0).sum()}")
print(f"99.9% quantile: {tslc.quantile(0.999)}")
print(f"Min: {tslc.min()}")
print(f"Max: {tslc.max()}")

## 7. time_since_last_change Outliers
Negative values: 0
99.9% quantile: 43224082.26200079
Min: 0
Max: 132447347


### Understanding the Feature

*time_since_last_change = c.committed_date - last_time*

Where:
- c.committed_date = current commit timestamp (UNIX seconds)
- last_time = timestamp of first parent commit

So the feature = time difference between consecutive commits.

This represents how much time passed between commits in a repo.

### Why Negative Values?

Meaning:

- Some commits appear to be ~4.6 days negative (-396,818 sec)
- Some commits appear to be ~77 days ahead (6.6M sec)

This is expected when real Git data is used.

Git timestamps can go backwards because:

#### (1) Rebased or rewritten history

During rebases, old commits appear “later” than newer ones.

*WHY?*

When you merge/rebase, Git does not reorder commits chronologically.
Instead, it preserves the logical order of development.

#### (2) Merge parents

You use only the first parent:

if c.parents:
    last_time = c.parents[0].committed_date


But merges may introduce non-linear time ordering.

#### (3) Clock drift

Different authors → different local machine clocks.

#### (4) Shallow clones or incomplete history

If the repo is shallow-fetched, parent commits may have weird timestamps.

None of this indicates your extraction is wrong.



## Binary Flag Integity

In [59]:
print("## 8. Binary Columns Integrity")
bin_cols = ["has_fix_kw", "has_bug_kw"]

for col in bin_cols:
    bad = df[~df[col].isin([0,1])]
    print(f"{col}: {len(bad)} invalid values")

## 8. Binary Columns Integrity


has_fix_kw: 0 invalid values
has_bug_kw: 0 invalid values


## Embedding Consistency Check

Ensure:

- no None
- all lists
- identical dimensionality

In [60]:
print(type(df.loc[0, 'code_embed']))
print(type(df.loc[0, 'msg_embed']))

print("## 9. Embedding Structural Checks")

# None count
print("code_embed None count:", df['code_embed'].isna().sum())
print("msg_embed None count:", df['msg_embed'].isna().sum())

# Check if all are lists
print("\nNon-list code_embed rows:", (~df['code_embed'].apply(lambda x: isinstance(x, list))).sum())
print("Non-list msg_embed rows:", (~df['msg_embed'].apply(lambda x: isinstance(x, list))).sum())

# Check dimensionality
dims = df['code_embed'].apply(lambda x: len(x) if isinstance(x, list) else None)
print("\nEmbedding dimensionality distribution:")
print(dims.value_counts().head())

<class 'list'>
<class 'list'>
## 9. Embedding Structural Checks
code_embed None count: 0
msg_embed None count: 0

Non-list code_embed rows: 0
Non-list msg_embed rows: 0

Embedding dimensionality distribution:
code_embed
768    58294
Name: count, dtype: int64


## Datetime Consistency

Check for:

- NaT values
- ordering sanity (commit should not be older than file's previous record)

In [61]:
print("## 10. Datetime Columns Audit")

date_cols = ["canonical_datetime"]

for col in date_cols:
    print(f"{col}: NaT count = {df[col].isna().sum()}")
    print(f"{col}: min = {df[col].min()}, max = {df[col].max()}")

## 10. Datetime Columns Audit
canonical_datetime: NaT count = 0
canonical_datetime: min = 2005-09-20 03:00:26+00:00, max = 2022-01-04 01:08:58+00:00


## Check Text Columns for Weirdness

Empty strings? Too short? Too long?

In [62]:
print("## 13. Text Field Checks")

if 'content' in df.columns:
    print("Empty content rows:", (df['content'].str.len() == 0).sum())
    print(df['content'].str.len().describe())

# Check 'methods' if it is a list column
if 'methods' in df.columns:
    # Use len() on the Python lists
    print("\nEmpty methods rows:", (df['methods'].apply(len) == 0).sum())
    print(df['methods'].apply(len).describe())

## 13. Text Field Checks


Empty content rows: 92
count    5.829400e+04
mean     4.426112e+03
std      1.293989e+04
min      0.000000e+00
25%      5.850000e+02
50%      1.462000e+03
75%      4.278750e+03
max      1.911019e+06
Name: content, dtype: float64


## Check For Impossible Values

In [63]:
print("## 14. Logical Consistency Checks")

# msg_len should match commit message length
if "msg_len" in df.columns:
    print("msg_len outliers (msg_len <= 0):", (df['msg_len'] <= 0).sum())

## 14. Logical Consistency Checks
msg_len outliers (msg_len <= 0): 0


## Check Recent Churn for Extreme Outliers

In [64]:
print("## 16. recent_churn Outlier Scan")
print(df['recent_churn'].describe())
print("99.9% quantile:", df['recent_churn'].quantile(0.999))

## 16. recent_churn Outlier Scan
count    58294.000000
mean        67.103150
std        443.234604
min          0.000000
25%          0.000000
50%          0.000000
75%         10.000000
max      31738.000000
Name: recent_churn, dtype: float64
99.9% quantile: 2534.675000000054


## Check Distribution of Code Activity Keywords

(todo, fixme, try/except/raise)

In [65]:
print("## 17. Keyword Column Distributions")
kw_cols = ["todo", "fixme", "try", "except", "raise"]

print(df[kw_cols].describe().T.to_markdown())

## 17. Keyword Column Distributions
|        |   count |      mean |       std |   min |   25% |   50% |   75% |   max |
|:-------|--------:|----------:|----------:|------:|------:|------:|------:|------:|
| todo   |   58294 | 0.0574845 |  0.646581 |     0 |     0 |     0 |     0 |   107 |
| fixme  |   58294 | 0.0124884 |  0.315674 |     0 |     0 |     0 |     0 |    52 |
| try    |   58294 | 0.23577   | 16.4646   |     0 |     0 |     0 |     0 |  3838 |
| except |   58294 | 0.29099   | 16.1217   |     0 |     0 |     0 |     0 |  3739 |
| raise  |   58294 | 0.220709  |  5.27098  |     0 |     0 |     0 |     0 |  1179 |


## Duplicate commit-message / code-embed lengths

Check uniformity:

In [66]:
code_duplicates = df['code_embed'].apply(len).value_counts().head()
msg_duplicates = df['msg_embed'].apply(len).value_counts().head()

print(code_duplicates)
print(msg_duplicates)

code_embed
768    58294
Name: count, dtype: int64
msg_embed
768    58294
Name: count, dtype: int64


So all your embeddings being length 768 means:

- The model you are using outputs a 768-dimensional vector
- The embedding extraction process worked for every commit
- No corrupted or empty embeddings
- No input missing (no None, no NaN, no empty list, etc.)

## Diff size sanity

Large mismatches could indicate:

- truncated diffs
- non-standard diff formatting
- metadata lines (prefix +++, ---, etc.)

In [67]:
(df["content"].str.count("\n") - df["loc_added"] - df["loc_deleted"]).describe()


count     58294.000000
mean        -35.570779
std        4772.074321
min     -811213.000000
25%           1.000000
50%           7.000000
75%          27.000000
max       20961.000000
dtype: float64

The discrepancy between content line counts and loc_added + loc_deleted is normal in real Git data.

Reasons:

1. Multi-line statements / code folding in diffs
2. Partial line changes counted in hunks
3. Files with removed lines only (content now shorter)
4. Large diffs in a single commit skew the stats

No indication of extraction errors — these are just properties of real commit diffs.